# Phase 3c: call figures and their counterparts in the reports

Plan: `plans/phase_3c.md` (revised before computation). Computation: `analysis/phase3c_counterparts.py`; outputs in `data/phase3c/`. Descriptive: no test, no register entry.

**Status (2026-09-23): the pre-specified rules do not measure the question.** The results are reported as computed; the rules were not repaired after seeing them, and the labelling sample was discarded.

Three reasons, shown in detail below: report sentences are often whole table blocks that contain a metric word, a forward-looking word and many numbers at once; the rounding tolerance lets an integer report figure such as USD 1bn match anything from 0.5 to 1.5 billion; and class B is met by almost every report, so C stays empty. Against the hand-read cases of phase 3b, three of six agree and three disagree.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parents[1] if Path.cwd().name == "roman" else Path.cwd()
OUT = ROOT / "notebooks" / "roman" / "data" / "phase3c"
pd.set_option("display.max_colwidth", 170)
shares = pd.read_csv(OUT / "shares.csv")
m = pd.read_csv(OUT / "matches.csv")
cols = ["cell", "candidates", "calls", "n_A", "n_B", "n_C", "share_A", "ci_A", "share_B", "ci_B", "share_C", "ci_C"]

## Main table 1: every candidate counts

In [ ]:
shares[(shares.setting == "primary") & (shares.level == "candidates")][cols].round(2)

## Main table 2: first mention of each thread counts

In [ ]:
shares[(shares.setting == "primary") & (shares.level == "threads")][cols].round(2).rename(columns={"candidates": "threads"})

Under the primary rules 52 % of UBS target candidates (36 % of UBS target threads) have the same figure in a forward-looking sentence of the same-quarter report, and none has no counterpart at all. The difference between the two tables reflects UBS repeating its main targets (13 billion savings, 15 % RoCET1, 14 % CET1) in many calls, and each repetition finds the same report sentence.

The JPMorgan guidance cell measures the design of the supplement, which contains no outlook: its A share of 17 % (7 % of threads) comes entirely from table dumps, as shown below, and is not interpreted.

## Sensitivity

In [ ]:
sens = shares[shares.cell.isin(["UBS target", "UBS guidance", "UBS other", "JPM guidance"])]
tab = sens.pivot_table(index="setting", columns=["level", "cell"], values="share_A").round(2)
tab

In [ ]:
base = tab.loc["primary"]
flag = (tab - base).abs() > 0.20
flag.loc[:, flag.any()].replace({True: "fragile", False: ""})

The UBS target share of A ranges from 0.27 (similarity instead of keyword) to 0.84 (window W2) for candidates and from 0.17 to 0.78 for threads. It moves by more than 20 percentage points with the wider windows W1 and W2 and with the similarity condition, and is therefore fragile; with exact tolerance it falls to 0.39 and 0.25, and with the horizon check to 0.35 and 0.22.

## Case check against the hand-read table of phase 3b

In [ ]:
cases = pd.DataFrame([
    (5214, "C2 integration expenses around 13bn by end-2026 (2024-Q1)", "not in any report (3b)"),
    (4762, "C2 integration expenses around 13bn by end-2026 (2023-Q4)", "not in any report (3b)"),
    (4794, "C4 AT1 plan up to 2bn in 2024 (2023-Q4)", "only in the call (3b)"),
    (4303, "C3 Basel III final roughly 5% (2023-Q3)", "no estimate in the 2023-Q3 report (3b)"),
    (4639, "C1 gross cost saves 13bn by end-2026 (2023-Q4)", "same figure in the 2023-Q4 report (3b)"),
    (767, "C5 First Republic restructuring about 2bn (event call)", "not restated in any supplement (3b)")],
    columns=["stmt_id", "case", "phase 3b reading"])
p = m[m.setting == "primary"][["stmt_id", "cls", "best_text", "best_location"]]
cases.merge(p, on="stmt_id", how="left")

Three of six cases agree with the hand-read table: the savings target in A, the First Republic restructuring costs in C, and the integration total of 2023-Q4 in B, which fits the class definition because the report gives quarterly integration costs but no total. Three disagree: the 13 billion integration-cost statement of 2024-Q1 lands in A because the savings target is also 13 billion, the AT1 plan lands in A through a CET1 sentence with a matching amount, and the Basel III statement of 2023-Q3 lands in B through a generic note, after phase 3 had filed it under non-core.

## Where the rules fail

In [ ]:
prim = m[m.setting == "primary"]
prim[(prim.bank == "JPM") & (prim.cls == "A")][["text", "best_text", "best_location"]].head(4)

Three failure sources are visible. JPMorgan report sentences are often whole table blocks (20 % of JPMorgan and 10 % of UBS report sentences are longer than 500 characters), which contain a metric word, a forward-looking word and many numbers at once. The tolerance of half a unit of the less precise figure lets an integer report figure such as USD 1bn match anything from 0.5 to 1.5 billion. And class B needs only a metric word and a forward-looking word anywhere in the report, which almost every report satisfies, so C stays empty for all keyword-based cells.

## Validation sample

In [ ]:
pd.read_csv(ROOT / "notebooks" / "roman" / "data" / "labelling" / "discarded_phase3c" / "counterparts_key.csv").groupby(["cls", "bank"]).size()

The sample of 45 blind pairs (fewer than the planned 60 because class C has only nine candidates) is discarded and not labelled, because labels would only confirm the known errors. It is kept for the record in `data/labelling/discarded_phase3c/`.

## Reading

Under the pre-specified rules the question cannot yet be answered: the share of call figures with a counterpart is dominated by coincidental number matches, table blocks and a class B that almost always applies, and it fails the case check in three of six cases. No repair was applied. The robust part is negative and matches phase 3b: the hand-read cases where the call carries figures that no report contains (total integration cost, AT1 plan, First Republic targets) are not recovered by the automatic rules.